# 06 · 进阶：CTE / 视图 / 索引 / 事务 / 安全

> 这一课把“从会写查询，到能维护真实系统”需要的底层概念补齐。它们不常出现在刷题里，却是你以后做后端、做数据工程时每天要面对的：怎么让复杂 SQL 可读、怎么让查询变快、怎么保证数据不被改坏、怎么防止被注入。
>
> 你是 C++ 竞赛出身，这一课的 B-Tree、事务 ACID、递归你会觉得很亲切——它们和数据结构、并发本质相通。

## 本课地图
1. CTE（WITH）：让复杂查询可读；递归 CTE 拆组织架构树
2. 视图 VIEW：给常用查询起个名字
3. 索引：B-Tree 直觉，什么时候加速、什么时候拖慢
4. 事务与 ACID：BEGIN / COMMIT / ROLLBACK
5. SQLite 的动态类型（和 MySQL/PG 的静态类型对比）
6. SQL 注入与参数化查询（安全底线）
7. 方言差异与“什么时候该迁到 MySQL/PostgreSQL”

In [1]:
# 统一环境：连接数据库（绝对路径，可反复运行）
import sqlite3
import pandas as pd

# 两个库都用绝对路径写死，避免“相对路径在哪个工作目录下”的坑
SHOP_DB = r"C:\Users\千禧\Doubao\chats\2026-09-23\new-chat\AI学习\06_SQL\data\shop.db"
SS_DB  = r"C:\Users\千禧\Doubao\chats\2026-09-23\new-chat\AI学习\06_SQL\data\superstore.db"

# 让 DataFrame 打印好看一点（你已经熟悉 Pandas，这里直接拿它当 SQL 的“显示层”）
pd.set_option("display.max_rows", 60)
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)

# 教学默认连 shop.db；需要 superstore 时再切
conn = sqlite3.connect(SHOP_DB)

def q(sql, conn=conn):
    """执行一条 SQL，把结果表直接返回成 Pandas DataFrame。"""
    return pd.read_sql_query(sql, conn)

print("已连接 shop.db。可用 q('SELECT ...') 跑查询。")
print("shop.db 表：", [r[0] for r in conn.execute(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name")])

已连接 shop.db。可用 q('SELECT ...') 跑查询。
shop.db 表： ['customers', 'employees', 'order_items', 'orders', 'products']


## 1. CTE（WITH）：让复杂查询可读

CTE = Common Table Expression，就是 `WITH 名字 AS (子查询)`。它和 04 课的派生表**功能一样**，但好处是：
- 可以定义多个、**互相引用**；
- 可以**递归**；
- 名字起得好，SQL 从上往下读像一段故事，不用括号嵌套地狱。

In [2]:
# 用 WITH 把“算客户消费 -> 再筛高价值客户”拆成两步，可读性吊打嵌套子查询
q('''
WITH cust_spend AS (
    SELECT c.customer_id, c.name, c.segment,
           SUM(oi.sales) AS total_spent,
           COUNT(DISTINCT o.order_id) AS n_orders
    FROM customers c
    JOIN orders o       ON o.customer_id = c.customer_id
    JOIN order_items oi ON oi.order_id = o.order_id
    GROUP BY c.customer_id, c.name, c.segment
)
SELECT *
FROM cust_spend
WHERE n_orders >= 8
ORDER BY total_spent DESC
LIMIT 10
''')

,customer_id,name,segment,total_spent,n_orders
0,239,李超,企业客户,278713.38,14
1,253,林梓涵,家庭办公,219832.72,13
2,160,马平,企业客户,213901.28,9
3,51,梁一诺,家庭办公,208011.66,15
4,149,林欣怡,个人客户,195391.93,15
5,123,何涛,个人客户,192931.33,15
6,242,罗秀英,个人客户,181633.85,11
7,97,马刚,企业客户,179549.14,11
8,166,吴超,家庭办公,179051.56,12
9,225,何梓涵,个人客户,176793.62,14


### 递归 CTE：拆组织架构树

`employees` 是“经理-员工”自引用的树形结构。要把整棵树从大老板往下摊平，普通 JOIN 写不动（不知道有几层）。**递归 CTE** 就是干这个的——和你在图论里写 DFS 一模一样：一个锚点 + 一个递归步骤。

In [3]:
q('''
WITH RECURSIVE org_tree AS (
    -- 锚点(Anchor)：树的根 = 没有上级的人
    SELECT employee_id, name, title, manager_id, 1 AS level,
           CAST(name AS TEXT) AS path
    FROM employees
    WHERE manager_id IS NULL

    UNION ALL

    -- 递归步骤：对上一层每个人，找出他的直接下属，level+1
    SELECT e.employee_id, e.name, e.title, e.manager_id, t.level + 1,
           t.path || ' > ' || e.name
    FROM employees e
    JOIN org_tree t ON e.manager_id = t.employee_id
)
SELECT level, path
FROM org_tree
ORDER BY path
LIMIT 15
''')

,level,path
0,1,张总
1,2,张总 > 市场总监
2,3,张总 > 市场总监 > 华东市场经理
3,4,张总 > 市场总监 > 华东市场经理 > 陈伟
4,3,张总 > 市场总监 > 华中市场经理
5,4,张总 > 市场总监 > 华中市场经理 > 刘娟
6,3,张总 > 市场总监 > 华北市场经理
7,4,张总 > 市场总监 > 华北市场经理 > 孙涛
8,4,张总 > 市场总监 > 华北市场经理 > 杨娜
9,3,张总 > 市场总监 > 华南市场经理


> 看懂这个你就懂了：锚点是 DFS 的起点，递归部分是“沿着 manager_id 边走边加深”，`UNION ALL` 把每一层结果拼起来。**任何“自引用树”（组织架构、菜单、评论楼中楼、分类目录）都用递归 CTE 摊平。**

## 2. 视图 VIEW：给常用查询起个名字

视图是“存下来的查询”，本身不存数据（像一个宏）。每次 `SELECT * FROM 视图名`，数据库就把定义展开执行。

In [4]:
# 创建一个视图：销售明细宽表。以后分析不用每次写那一堆 JOIN
cur = conn.cursor()
cur.execute('DROP VIEW IF EXISTS v_sales_detail')
cur.execute('''
CREATE VIEW v_sales_detail AS
SELECT o.order_id, o.order_date, o.region, o.city,
       c.name AS customer_name, c.segment,
       p.product_name, p.category, p.subcategory,
       oi.quantity, oi.discount, oi.sales, oi.profit
FROM orders o
JOIN customers c    ON c.customer_id = o.customer_id
JOIN order_items oi ON oi.order_id = o.order_id
JOIN products p     ON p.product_id = oi.product_id
''')
conn.commit()
print("视图 v_sales_detail 已创建")

# 之后分析就像查一张表一样简单
q('''SELECT category, ROUND(SUM(profit),1) AS 总利润
     FROM v_sales_detail GROUP BY category ORDER BY 总利润 DESC''')

视图 v_sales_detail 已创建


,category,总利润
0,科技产品,5774662.8
1,家具,2346428.4
2,办公用品,112215.9


> 视图的价值：**把复杂 JOIN 藏起来，给分析者一个干净的“宽表”接口**。BI 工具、报表基本都建立在视图之上。注意它不存数据，所以不会加速查询，只是简化写法。

## 3. 索引：B-Tree 直觉，什么时候加速

你建库脚本里已经有 4 个索引。索引是怎么加速的？

> 类比：书最后那本“关键词目录”。没有它，找一个关键词得逐页翻（全表扫描 O(n)）；有了目录，按拼音/笔画几跳就找到（B-Tree O(log n)）。

数据库默认索引结构是 **B-Tree**（平衡多路搜索树）：有序、对数级查找。`EXPLAIN QUERY PLAN` 能告诉你一条查询有没有走索引。

In [5]:
# 没有索引时：全表扫描，扫 6407 行
q('''EXPLAIN QUERY PLAN
SELECT * FROM order_items WHERE product_id = 54''')

,id,parent,notused,detail
0,3,0,61,SEARCH order_items USING INDEX idx_items_produ...


In [6]:
# 我们建库时建了 idx_items_product，按 product_id 查应该走索引
# 看 result 里有没有 "USING INDEX" / "SEARCH"
q('''EXPLAIN QUERY PLAN
SELECT oi.* FROM order_items oi WHERE oi.product_id = 54''')

,id,parent,notused,detail
0,3,0,61,SEARCH oi USING INDEX idx_items_product (produ...


索引什么时候**帮倒忙**？这是关键：

- **写多读少的表别乱建索引**：每次 INSERT/DELETE/UPDATE，索引都要同步维护，写变慢；
- **小表不用建**：全表扫描本来就快，索引反而多一层跳转；
- **区分度低的列没用**：比如 `gender` 只有男女两个值，建索引和全表扫描差不多；
- 索引占磁盘空间。

> 经验：给 **WHERE / JOIN / ORDER BY / GROUP BY 高频用到的列**建索引，尤其是外键列。06 课这个直觉面试够用了，调优深入看 Use The Index, Luke。

## 4. 事务与 ACID

事务(transaction)把一组操作打包成“要么全成、要么全败”。银行转账是经典例子：扣 A 加 B，中间断电不能只扣不加。

In [7]:
# 演示事务：故意制造一次失败，然后 ROLLBACK，看数据没变
# 用一张临时表，避免动到你的业务数据
cur = conn.cursor()
cur.execute("CREATE TABLE IF NOT EXISTS demo_account(name TEXT, balance REAL)")
cur.execute("DELETE FROM demo_account")
cur.execute("INSERT INTO demo_account VALUES ('A', 100), ('B', 0)")
conn.commit()

print("转账前：", cur.execute("SELECT * FROM demo_account").fetchall())

# 开始事务：A 转给 B 50
cur.execute("BEGIN")
cur.execute("UPDATE demo_account SET balance = balance - 50 WHERE name='A'")
cur.execute("UPDATE demo_account SET balance = balance + 50 WHERE name='B'")
# 突然出错 -> ROLLBACK，两步都撤销
cur.execute("ROLLBACK")
print("ROLLBACK 后：", cur.execute("SELECT * FROM demo_account").fetchall(),
      "（A 没被扣，符合预期）")

# 再来一次，这次 COMMIT
cur.execute("BEGIN")
cur.execute("UPDATE demo_account SET balance = balance - 50 WHERE name='A'")
cur.execute("UPDATE demo_account SET balance = balance + 50 WHERE name='B'")
conn.commit()
print("COMMIT 后：", cur.execute("SELECT * FROM demo_account").fetchall())
cur.execute("DROP TABLE demo_account")
conn.commit()

转账前： [('A', 100.0), ('B', 0.0)]
ROLLBACK 后： [('A', 100.0), ('B', 0.0)] （A 没被扣，符合预期）
COMMIT 后： [('A', 50.0), ('B', 50.0)]


ACID 是事务的四大性质（面试爱考缩写）：

- **A 原子性 Atomicity**：一组操作要么全做要么全不做（上面 ROLLBACK 演示的）；
- **C 一致性 Consistency**：事务前后数据库约束不被破坏；
- **I 隔离性 Isolation**：并发事务互不干扰；
- **D 持久性 Durability**：COMMIT 后即使断电也不丢。

> SQLite 是**文件级锁**，适合单机/嵌入式；MySQL/PostgreSQL 用更复杂的并发控制，才能扛多用户同时写。这也是“什么时候该换库”的理由之一。

## 5. SQLite 的动态类型（一个重要区别）

大多数数据库是“强类型”：列声明了 INTEGER 就只能存整数。SQLite 是**动态类型**：它按“值本身”判断类型，列声明只是“亲和性提示”。

In [8]:
# SQLite 里即使列声明 INTEGER，也能塞进去文本（动态类型）
cur = conn.cursor()
cur.execute("CREATE TABLE IF NOT EXISTS demo_type(x INTEGER)")
cur.execute("DELETE FROM demo_type")
cur.execute("INSERT INTO demo_type VALUES (123)")
cur.execute("INSERT INTO demo_type VALUES ('我是文本')")   # 居然能插
print("类型亲和性演示：", cur.execute("SELECT x, typeof(x) FROM demo_type").fetchall())
cur.execute("DROP TABLE demo_type")
conn.commit()

类型亲和性演示： [(123, 'integer'), ('我是文本', 'text')]


> 这就是为什么说“SQLite 是学习工具，不是强约束的生产库”。迁到 PostgreSQL 时，类型约束会严格起来——这是优点。写迁移脚本时别依赖 SQLite 的宽容。

## 6. SQL 注入与参数化查询（安全底线）

你在 00 课见过 `?` 占位。这里讲为什么——因为**字符串拼接 SQL 会被注入攻击**。

In [9]:
# 反面教材：用 Python 字符串拼接用户输入 -> 注入漏洞
bad_user_input = "华东' OR '1'='1"   # 恶意输入
bad_sql = f"SELECT * FROM customers WHERE region = '{bad_user_input}'"
print("危险的拼接 SQL：", bad_sql)
print("-> 条件变成 region='华东' OR '1'='1'，永远为真，全表泄露！")

# 正确姿势：参数化，? 占位，由库驱动负责转义
cur = conn.cursor()
cur.execute("SELECT COUNT(*) FROM customers WHERE region = ?", (bad_user_input,))
print("参数化后查出来：", cur.fetchone()[0], "行（恶意输入被当成普通字符串，没注入成功）")

危险的拼接 SQL： SELECT * FROM customers WHERE region = '华东' OR '1'='1'
-> 条件变成 region='华东' OR '1'='1'，永远为真，全表泄露！
参数化后查出来： 0 行（恶意输入被当成普通字符串，没注入成功）


> 铁律：**永远不要用 f-string / % 拼接用户输入进 SQL**，一律 `?`（SQLite）/ `%s`（psycopg）占位传参。这是 OWASP 排名第一的 Web 漏洞，记住它能救命。

## 7. 方言差异与迁移时机

学完本模块，你已经能用 SQLite 做绝大多数分析。什么时候该认真学 MySQL/PostgreSQL？

- 要做**真实后端/多用户并发**：SQLite 写锁太粗，这时上 PostgreSQL/MySQL；
- 公司岗位要求：互联网后端常考 MySQL，数据/算法岗常考 PostgreSQL；
- 需要高级特性：PostgreSQL 的 JSON、GIS、丰富函数；MySQL 的生态。

语法 90% 相同，差异主要在：分页（SQLite `LIMIT n OFFSET m`，MySQL `LIMIT m,n`，PG 同 SQLite）、日期函数、自增主键、字符串拼接等。**先把本模块的“通用 SQL”学透，方言差异是查文档的事，不是重新学一门语言。**

## 本课小结

- CTE（WITH）让复杂查询可读、可递归；递归 CTE 拆组织树，本质是 DFS。
- 视图 = 存下来的查询，简化宽表接口，不加速。
- 索引是 B-Tree，加速 WHERE/JOIN/ORDER BY；但写多、表小、区分度低时别建。
- 事务 ACID：BEGIN/COMMIT/ROLLBACK，一组操作要么全成要么全败。
- SQLite 动态类型、文件级锁；生产高并发迁 MySQL/PG。
- 永远参数化查询，杜绝 SQL 注入。

## 来源与延伸

- SQLite 官方：WITH（CTE/递归）https://www.sqlite.org/lang_with.html ；事务 https://www.sqlite.org/lang_transaction.html
- 索引性能圣经 Use The Index, Luke：https://use-the-index-luke.com/
- PostgreSQL Tutorial（窗口函数/CTE/索引系统教程）：https://www.postgresqltutorial.com/
- MySQL Tutorial：https://www.mysqltutorial.org/
- 真实练手库 Chinook（2.6k stars，截至 2026-10，多音乐表结构）：https://github.com/lerocha/chinook-database
- Northwind-SQLite3（621 stars，截至 2026-10，经典 Northwind 的 SQLite 版）：https://github.com/jpwhite3/northwind-SQLite3
- MySQL 官方 Sakila 示例库文档：https://dev.mysql.com/doc/sakila/en/